# Emotion-Aware Multi-Artist Generative Art — Full Pipeline

**Project:** Emotion-Aware Multi-Artist Generative Art with AI Interpretation

**Artists:** Vincent van Gogh | Claude Monet | Leonardo da Vinci

**Pipeline stages covered in this notebook:**

1. Environment setup and dataset download
2. Exploratory Data Analysis (EDA) on the full dataset
3. Filter to the three target artists
4. Feature Engineering (pixel stats, metadata, derived features)
5. Emotion Labeling — Color-based heuristics
6. Emotion Labeling — CLIP-based similarity (optional, requires GPU)
7. GAN Preprocessing — resize to 256x256, normalize to [-1, 1], augmentation
8. Save GAN-ready dataset and labeled DataFrame

## Part 1 — Environment Setup

In [ ]:
!pip install pandas numpy matplotlib seaborn pillow scikit-learn kagglehub -q

import os, re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

print("All libraries imported.")

## Part 2 — Download Dataset

In [ ]:
import kagglehub

path         = kagglehub.dataset_download("ikarus777/best-artworks-of-all-time")
csv_path     = os.path.join(path, "artists.csv")
resized_path = os.path.join(path, "resized", "resized")

print("Dataset path :", path)
print("CSV path     :", csv_path)
print("Images path  :", resized_path)

## Part 3 — EDA on Full Dataset

In [ ]:
artists_meta = pd.read_csv(csv_path)
print("CSV shape:", artists_meta.shape)
print("Columns  :", artists_meta.columns.tolist())
artists_meta.head()

In [ ]:
print("Missing values:")
print(artists_meta.isnull().sum())

In [ ]:
all_files = [f for f in os.listdir(resized_path) if f.endswith(".jpg")]
print("Total images in dataset:", len(all_files))

In [ ]:
# Count images per artist across the full dataset
artist_counts = {}
for fname in all_files:
    parts = fname.replace(".jpg", "").rsplit("_", 1)
    name  = parts[0] if len(parts) == 2 and parts[1].isdigit() else fname.replace(".jpg", "")
    artist_counts[name] = artist_counts.get(name, 0) + 1

df_counts = pd.DataFrame(list(artist_counts.items()), columns=["Artist", "Image_Count"])
df_counts = df_counts.sort_values("Image_Count", ascending=False).reset_index(drop=True)

plt.figure(figsize=(14, 7))
sns.barplot(x="Image_Count", y="Artist", data=df_counts.head(20), palette="viridis")
plt.title("Top 20 Artists by Number of Images")
plt.xlabel("Number of Images")
plt.tight_layout()
plt.show()

print("Total artists:", len(df_counts))
df_counts.head(10)

In [ ]:
# Inspect a random 9-image sample from the full dataset
plt.figure(figsize=(12, 12))
sample_files = random.sample(all_files, 9)
for i, fname in enumerate(sample_files):
    img = Image.open(os.path.join(resized_path, fname))
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title("_".join(fname.split("_")[:-1]), fontsize=7)
    plt.axis("off")
plt.suptitle("Random Sample — Full Dataset", fontsize=14)
plt.tight_layout()
plt.show()

## Part 4 — Filter to Target Artists

We keep only the three artists relevant to this project:
- Vincent van Gogh
- Claude Monet
- Leonardo da Vinci

In [ ]:
TARGET_ARTISTS = ["Vincent_van_Gogh", "Claude_Monet", "Leonardo_da_Vinci"]

records = []
for fname in all_files:
    parts = fname.replace(".jpg", "").rsplit("_", 1)
    if len(parts) == 2 and parts[1].isdigit():
        artist_raw = parts[0]
        artwork_id = int(parts[1])
    else:
        continue
    if artist_raw not in TARGET_ARTISTS:
        continue
    records.append({
        "filename"   : fname,
        "filepath"   : os.path.join(resized_path, fname),
        "artist_raw" : artist_raw,
        "artist_name": artist_raw.replace("_", " "),
        "artwork_id" : artwork_id,
    })

df = pd.DataFrame(records)
print("Filtered dataset shape:", df.shape)
print(df["artist_name"].value_counts())

In [ ]:
# Show 3 sample images per artist
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle("Sample Images — Three Target Artists", fontsize=14, fontweight="bold")

for row_idx, artist in enumerate(TARGET_ARTISTS):
    subset = df[df["artist_raw"] == artist].sample(3, random_state=row_idx)
    for col_idx, (_, row) in enumerate(subset.iterrows()):
        img = Image.open(row["filepath"])
        axes[row_idx][col_idx].imshow(img)
        axes[row_idx][col_idx].set_title(row["artist_name"], fontsize=8)
        axes[row_idx][col_idx].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Per-artist image count bar chart
plt.figure(figsize=(7, 4))
df["artist_name"].value_counts().plot(kind="bar", color=["steelblue", "seagreen", "tomato"])
plt.title("Image Count per Target Artist")
plt.ylabel("Number of Images")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Part 5 — Merge artists.csv Metadata

In [ ]:
artists_meta["artist_raw"] = artists_meta["name"].str.replace(" ", "_")

meta_slim = artists_meta[["artist_raw", "years", "genre", "nationality", "paintings"]].copy()
df = df.merge(meta_slim, on="artist_raw", how="left")

# Parse birth/death year
def parse_years(s):
    if pd.isna(s): return pd.Series([None, None, None])
    nums = re.findall(r"\d{4}", str(s))
    birth = int(nums[0]) if len(nums) >= 1 else None
    death = int(nums[1]) if len(nums) >= 2 else None
    age   = (death - birth) if birth and death else None
    return pd.Series([birth, death, age])

df[["birth_year", "death_year", "age_at_death"]] = df["years"].apply(parse_years)

print("Shape after metadata merge:", df.shape)
df[["artist_name", "genre", "nationality", "birth_year", "death_year"]]\
    .drop_duplicates("artist_name")

## Part 6 — Feature Engineering — Pixel-Level Statistics

We extract 14 numerical features from each image's raw pixels.

| Feature | Description |
|---------|-------------|
| img_width, img_height | Dimensions in pixels |
| aspect_ratio | width / height |
| mean_r, mean_g, mean_b | Average value per RGB channel |
| std_r, std_g, std_b | Standard deviation per RGB channel |
| brightness | Mean of grayscale-converted image |
| contrast | Std of grayscale-converted image |
| saturation | Mean of S channel in HSV |
| hue_mean | Mean of H channel in HSV |
| colorfulness | Hasler and Suesstrunk (2003) colorfulness metric |

In [ ]:
def extract_pixel_features(filepath):
    try:
        img_rgb = Image.open(filepath).convert("RGB")
        arr     = np.array(img_rgb, dtype=np.float32)
        h, w, _ = arr.shape
        r, g, b = arr[:,:,0], arr[:,:,1], arr[:,:,2]
        gray    = 0.299*r + 0.587*g + 0.114*b
        hsv     = np.array(Image.open(filepath).convert("HSV"), dtype=np.float32)
        rg      = r - g
        yb      = 0.5*(r + g) - b
        cf      = np.sqrt(rg.std()**2 + yb.std()**2) + 0.3*np.sqrt(rg.mean()**2 + yb.mean()**2)
        return {
            "img_width"   : w,
            "img_height"  : h,
            "aspect_ratio": round(w/h, 4),
            "mean_r"      : round(r.mean(), 3),
            "mean_g"      : round(g.mean(), 3),
            "mean_b"      : round(b.mean(), 3),
            "std_r"       : round(r.std(), 3),
            "std_g"       : round(g.std(), 3),
            "std_b"       : round(b.std(), 3),
            "brightness"  : round(gray.mean(), 3),
            "contrast"    : round(gray.std(), 3),
            "saturation"  : round(hsv[:,:,1].mean(), 3),
            "hue_mean"    : round(hsv[:,:,0].mean(), 3),
            "colorfulness": round(cf, 3),
        }
    except Exception:
        return None


print("Extracting pixel features for all", len(df), "images...")
feat_rows = []
for _, row in df.iterrows():
    f = extract_pixel_features(row["filepath"])
    if f:
        f["filename"] = row["filename"]
        feat_rows.append(f)

feat_df = pd.DataFrame(feat_rows)
df = df.merge(feat_df, on="filename", how="inner")
print("Done. Shape:", df.shape)

In [ ]:
# Derived features
df["is_portrait"]     = (df["img_height"] > df["img_width"]).astype(int)
df["is_colorful"]     = (df["colorfulness"] > df["colorfulness"].median()).astype(int)
df["color_temp"]      = df.apply(lambda r: "Warm" if r["mean_r"] > r["mean_b"] else "Cool", axis=1)
df["dominant_channel"]= df.apply(
    lambda r: "Red" if r["mean_r"] >= r["mean_g"] and r["mean_r"] >= r["mean_b"]
    else ("Green" if r["mean_g"] >= r["mean_b"] else "Blue"), axis=1
)

print("Derived features added.")
df[["filename", "artist_name", "brightness", "colorfulness",
    "is_portrait", "color_temp", "dominant_channel"]].head(8)

In [ ]:
# Per-artist pixel stats comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Pixel Statistics by Artist", fontsize=14, fontweight="bold")

for ax, metric, color in zip(axes,
    ["brightness", "saturation", "colorfulness"],
    ["coral", "steelblue", "mediumseagreen"]):
    df.boxplot(column=metric, by="artist_name", ax=ax)
    ax.set_title(metric.capitalize())
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Pixel Statistics by Artist", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — pixel features
corr_cols = ["brightness", "contrast", "saturation", "colorfulness",
             "mean_r", "mean_g", "mean_b", "aspect_ratio", "is_portrait", "is_colorful"]
corr_cols = [c for c in corr_cols if c in df.columns]

plt.figure(figsize=(10, 7))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, square=True, linewidths=0.4)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## Part 7 — Emotion Labeling — Method 1: Color-Based Heuristics

We assign one of three emotion labels to each image using pixel statistics.

The logic is grounded in color psychology research:

| Emotion | Signal |
|---------|--------|
| Joy | High brightness AND high colorfulness AND warm color temperature |
| Sadness | Low brightness AND cool color temperature (blue dominant) |
| Calm | Everything in between — moderate brightness and saturation |

This is a heuristic approach. It is fast, requires no GPU, and gives reasonable labels for training.

In [ ]:
def color_based_emotion(row):
    """
    Assign an emotion label using brightness, colorfulness, and color temperature.
    Thresholds are based on the dataset distribution (percentile-driven).
    """
    b  = row["brightness"]
    cf = row["colorfulness"]
    ct = row["color_temp"]

    if b >= 160 and cf >= 40 and ct == "Warm":
        return "Joy"
    elif b <= 100 and ct == "Cool":
        return "Sadness"
    else:
        return "Calm"

df["emotion_color"] = df.apply(color_based_emotion, axis=1)

print("Emotion distribution (color-based):")
print(df["emotion_color"].value_counts())

In [ ]:
# Emotion distribution per artist
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Emotion Distribution by Artist (Color-Based)", fontsize=13, fontweight="bold")

palette = {"Joy": "gold", "Calm": "steelblue", "Sadness": "mediumpurple"}

for ax, artist in zip(axes, [a.replace("_", " ") for a in TARGET_ARTISTS]):
    subset = df[df["artist_name"] == artist]
    counts = subset["emotion_color"].value_counts()
    counts.plot(kind="bar", ax=ax,
                color=[palette.get(e, "gray") for e in counts.index])
    ax.set_title(artist, fontsize=9)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize 3 images per emotion
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle("Sample Images by Emotion (Color-Based)", fontsize=14, fontweight="bold")

for row_idx, emotion in enumerate(["Joy", "Calm", "Sadness"]):
    subset = df[df["emotion_color"] == emotion].sample(3, random_state=42)
    for col_idx, (_, img_row) in enumerate(subset.iterrows()):
        img = Image.open(img_row["filepath"])
        axes[row_idx][col_idx].imshow(img)
        axes[row_idx][col_idx].set_title(
            f"{emotion} | {img_row['artist_name'].split(' ')[-1]}", fontsize=8
        )
        axes[row_idx][col_idx].axis("off")

plt.tight_layout()
plt.show()

## Part 8 — Emotion Labeling — Method 2: CLIP-Based Similarity (Optional)

CLIP (Contrastive Language-Image Pretraining) is a model from OpenAI that understands
both images and text. We feed it each image and three text prompts:

- 'a joyful, bright, colorful painting'
- 'a calm, peaceful, serene painting'
- 'a sad, dark, melancholic painting'

CLIP returns a similarity score for each prompt. We assign the label with the highest score.

This method is more semantically accurate than color heuristics but requires a GPU and more time.

**How to enable:** Set RUN_CLIP = True below. Make sure you are using a Colab GPU runtime.

In [ ]:
RUN_CLIP = False  # Set to True to run CLIP-based labeling

if RUN_CLIP:
    !pip install open_clip_torch -q
    import torch
    import open_clip

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)

    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="openai"
    )
    model = model.to(device).eval()
    tokenizer = open_clip.get_tokenizer("ViT-B-32")

    EMOTION_PROMPTS = [
        "a joyful, bright, colorful painting",
        "a calm, peaceful, serene painting",
        "a sad, dark, melancholic painting",
    ]
    EMOTION_NAMES = ["Joy", "Calm", "Sadness"]

    text_tokens = tokenizer(EMOTION_PROMPTS).to(device)
    with torch.no_grad():
        text_features = model.encode_text(text_tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    def clip_emotion(filepath):
        try:
            img_tensor = preprocess(Image.open(filepath)).unsqueeze(0).to(device)
            with torch.no_grad():
                img_features = model.encode_image(img_tensor)
                img_features = img_features / img_features.norm(dim=-1, keepdim=True)
                scores = (img_features @ text_features.T).squeeze(0)
            return EMOTION_NAMES[scores.argmax().item()]
        except Exception:
            return None

    print("Running CLIP emotion labeling on", len(df), "images...")
    df["emotion_clip"] = df["filepath"].apply(clip_emotion)
    print("CLIP emotion distribution:")
    print(df["emotion_clip"].value_counts())

else:
    df["emotion_clip"] = None
    print("CLIP labeling skipped. Set RUN_CLIP = True to enable it.")

In [ ]:
# Choose which emotion label to use for training
# If CLIP was run, use emotion_clip. Otherwise fall back to color-based.

df["emotion_final"] = df["emotion_clip"].fillna(df["emotion_color"])

print("Final emotion label distribution:")
print(df["emotion_final"].value_counts())

print("\nMethod used per image:")
clip_count  = df["emotion_clip"].notna().sum()
color_count = df["emotion_clip"].isna().sum()
print(f"  CLIP-based  : {clip_count} images")
print(f"  Color-based : {color_count} images")

## Part 9 — GAN Preprocessing

Before training a GAN, images must be:

1. **Resized to 256x256** — fixed input size required by the GAN architecture
2. **Converted to RGB** — ensures no grayscale or RGBA images slip through
3. **Normalized to [-1, 1]** — GANs use tanh activation in the generator output layer,
   which produces values in [-1, 1]. Matching this range during training stabilizes learning.

Formula: `pixel_normalized = (pixel / 127.5) - 1.0`

We also apply optional data augmentation to reduce overfitting:
- Horizontal flip (50% probability)
- Small random rotation (+/- 15 degrees)
- Slight color jitter

In [ ]:
import torchvision.transforms as T
from torchvision.utils import save_image
import torch

GAN_IMAGE_SIZE = 256

# Standard GAN preprocessing transform
gan_transform = T.Compose([
    T.Resize((GAN_IMAGE_SIZE, GAN_IMAGE_SIZE)),
    T.ToTensor(),                             # [0, 255] -> [0.0, 1.0]
    T.Normalize(mean=[0.5, 0.5, 0.5],
                std=[0.5, 0.5, 0.5]),          # [0.0, 1.0] -> [-1.0, 1.0]
])

# Augmentation transform (used only during GAN training, not saved to disk)
aug_transform = T.Compose([
    T.Resize((GAN_IMAGE_SIZE, GAN_IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

print("GAN transforms defined.")
print("Target image size:", GAN_IMAGE_SIZE, "x", GAN_IMAGE_SIZE)
print("Normalization range: [-1, 1]")

In [ ]:
# Verify preprocessing on a single image
sample_path = df.iloc[0]["filepath"]
raw_img     = Image.open(sample_path).convert("RGB")
tensor_img  = gan_transform(raw_img)

print("Original size :", raw_img.size)
print("Tensor shape  :", tensor_img.shape)   # Should be [3, 256, 256]
print("Tensor min    :", round(tensor_img.min().item(), 4))   # Should be close to -1
print("Tensor max    :", round(tensor_img.max().item(), 4))   # Should be close to +1

# Visualize before / after
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(raw_img)
axes[0].set_title("Original")
axes[0].axis("off")

# Denormalize for display: pixel = (tensor + 1) / 2
display_tensor = (tensor_img + 1) / 2
display_img    = T.ToPILImage()(display_tensor)
axes[1].imshow(display_img)
axes[1].set_title("After GAN Preprocessing (256x256, RGB)")
axes[1].axis("off")

plt.suptitle("GAN Preprocessing Verification", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Save preprocessed images to disk, organized by artist and emotion
OUTPUT_DIR = "/content/gan_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

saved = 0
skipped = 0

for _, row in df.iterrows():
    artist_dir  = os.path.join(OUTPUT_DIR, row["artist_raw"], row["emotion_final"])
    os.makedirs(artist_dir, exist_ok=True)

    out_path = os.path.join(artist_dir, row["filename"])
    if os.path.exists(out_path):
        skipped += 1
        continue

    try:
        img    = Image.open(row["filepath"]).convert("RGB")
        tensor = gan_transform(img)
        # Denormalize back to [0, 1] for saving as PNG
        save_image((tensor + 1) / 2, out_path)
        saved += 1
    except Exception:
        skipped += 1

print(f"Preprocessing complete.")
print(f"Saved : {saved} images")
print(f"Skipped: {skipped} images")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Structure: gan_dataset / artist / emotion / image.jpg")

In [ ]:
# Verify the folder structure
for artist in os.listdir(OUTPUT_DIR):
    artist_path = os.path.join(OUTPUT_DIR, artist)
    if os.path.isdir(artist_path):
        for emotion in os.listdir(artist_path):
            emotion_path = os.path.join(artist_path, emotion)
            count = len(os.listdir(emotion_path))
            print(f"{artist:30s} | {emotion:8s} | {count} images")

## Part 10 — Encode, Normalize, and Save Final DataFrame

In [ ]:
# Label encode artist name and emotion
le = LabelEncoder()
df["artist_enc"]  = le.fit_transform(df["artist_name"])
df["emotion_enc"] = le.fit_transform(df["emotion_final"])

# One-hot encode emotion for direct use in conditioning
df = pd.get_dummies(df, columns=["emotion_final"], prefix="emotion")

# Normalize numerical pixel features to [0, 1]
num_cols = ["brightness", "contrast", "saturation", "colorfulness",
            "mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
            "aspect_ratio", "hue_mean"]
num_cols = [c for c in num_cols if c in df.columns]

scaler = MinMaxScaler()
df[[f"{c}_norm" for c in num_cols]] = scaler.fit_transform(df[num_cols])

# Drop helper columns not needed downstream
drop_cols = [c for c in ["filepath", "artist_raw", "years", "emotion_color", "emotion_clip"]
             if c in df.columns]
df_final = df.drop(columns=drop_cols)

# Save
csv_out = "/content/artworks_gan_ready.csv"
df_final.to_csv(csv_out, index=False)

print("Saved:", csv_out)
print("Shape:", df_final.shape)
print("\nAll columns:")
print(df_final.columns.tolist())

---
# Part 11 — StyleGAN2-ADA Transfer Learning

We now use **StyleGAN2-ADA** with Transfer Learning to generate new artworks
inspired by our three artists.

**Why StyleGAN2-ADA?**

| Property | Benefit for our project |
|----------|-------------------------|
| Pre-trained on FFHQ (70k faces) | Strong low-level features already learned — textures, edges, colors |
| ADA (Adaptive Discriminator Augmentation) | Prevents overfitting on small datasets like ours |
| Style-based architecture | We can inject emotion conditioning into the style vectors (W space) |
| Fine-tuning from checkpoint | We don't train from scratch — we start from a strong base |

**Pipeline:**
```
Pre-trained StyleGAN2 checkpoint (FFHQ)
        ↓
Fine-tune on our dataset (Van Gogh + Monet + Leonardo)
        ↓
Emotion vector injected into W latent space
        ↓
Generated image in the style of the selected artist
        ↓
Claude API describes the generated artwork
```

## Part 11.1 — Install StyleGAN2-ADA

In [ ]:
# Clone the official StyleGAN2-ADA PyTorch repository
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git

# Install required packages
!pip install ninja torch torchvision torchaudio -q
!pip install click requests tqdm pyspng imageio-ffmpeg -q

import sys
sys.path.insert(0, '/content/stylegan2-ada-pytorch')

import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — please switch to GPU runtime")

## Part 11.2 — Download Pre-Trained StyleGAN2 Checkpoint

We download the FFHQ-256 checkpoint — pre-trained on 70,000 human face images.

Even though faces are different from paintings, the low-level features
(edges, textures, color gradients) transfer well to artwork generation.
This is the core idea of **Transfer Learning**.

In [ ]:
import os

CHECKPOINT_DIR = "/content/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

PRETRAINED_URL  = "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/transfer-learning-source-nets/ffhq256.pkl"
PRETRAINED_PATH = os.path.join(CHECKPOINT_DIR, "ffhq256.pkl")

if not os.path.exists(PRETRAINED_PATH):
    print("Downloading pre-trained checkpoint (~200 MB)...")
    !wget -q --show-progress -O {PRETRAINED_PATH} {PRETRAINED_URL}
    print("Download complete.")
else:
    print("Checkpoint already exists:", PRETRAINED_PATH)

print("Checkpoint size:", round(os.path.getsize(PRETRAINED_PATH) / 1e6, 1), "MB")

## Part 11.3 — Prepare Dataset in StyleGAN2 Format

StyleGAN2-ADA requires images to be stored as a `.zip` archive
using its own `dataset_tool.py` script.

We use the preprocessed images from **Part 9** (`/content/gan_dataset/`).
We flatten all artist folders into a single directory first,
then convert to `.zip`.

In [ ]:
import shutil
from PIL import Image

# Flatten all artist/emotion subdirectories into one flat folder
FLAT_DIR = "/content/stylegan_flat"
os.makedirs(FLAT_DIR, exist_ok=True)

GAN_DATASET_DIR = "/content/gan_dataset"
count = 0

for artist in os.listdir(GAN_DATASET_DIR):
    artist_path = os.path.join(GAN_DATASET_DIR, artist)
    if not os.path.isdir(artist_path):
        continue
    for emotion in os.listdir(artist_path):
        emotion_path = os.path.join(artist_path, emotion)
        if not os.path.isdir(emotion_path):
            continue
        for fname in os.listdir(emotion_path):
            if not fname.endswith((".jpg", ".png")):
                continue
            src  = os.path.join(emotion_path, fname)
            dst  = os.path.join(FLAT_DIR, f"{artist}_{emotion}_{fname}")
            # Ensure image is 256x256 RGB PNG
            img  = Image.open(src).convert("RGB").resize((256, 256))
            img.save(dst.replace(".jpg", ".png"))
            count += 1

print(f"Flattened {count} images to: {FLAT_DIR}")

In [ ]:
# Convert flat image folder to StyleGAN2 .zip dataset format
DATASET_ZIP = "/content/artworks_stylegan.zip"

!python /content/stylegan2-ada-pytorch/dataset_tool.py \
    --source={FLAT_DIR} \
    --dest={DATASET_ZIP} \
    --width=256 \
    --height=256

print("Dataset zip created:", DATASET_ZIP)
print("Size:", round(os.path.getsize(DATASET_ZIP) / 1e6, 1), "MB")

## Part 11.4 — Fine-Tune StyleGAN2-ADA on Our Artwork Dataset

We start from the FFHQ checkpoint and fine-tune on our artwork dataset.

Key training arguments explained:

| Argument | Value | Reason |
|----------|-------|--------|
| `--resume` | ffhq256.pkl | Start from pre-trained weights — Transfer Learning |
| `--freezed` | 4 | Freeze first 4 layers — keep low-level features from pre-training |
| `--aug` | ada | Adaptive augmentation — prevents overfitting on small dataset |
| `--mirror` | 1 | Horizontal flip augmentation — doubles effective dataset size |
| `--kimg` | 200 | Train for 200k images seen — enough for fine-tuning on Colab |
| `--snap` | 10 | Save checkpoint every 10k images |
| `--metrics` | none | Skip FID calculation to save time during training |

> **Expected training time on Colab T4:** ~3-5 hours for 200 kimg.
> You can stop earlier (at 50-100 kimg) and use the latest checkpoint.

In [ ]:
RESULTS_DIR = "/content/stylegan_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

!python /content/stylegan2-ada-pytorch/train.py \
    --outdir={RESULTS_DIR} \
    --data={DATASET_ZIP} \
    --gpus=1 \
    --cfg=paper256 \
    --resume={PRETRAINED_PATH} \
    --freezed=4 \
    --aug=ada \
    --mirror=1 \
    --kimg=200 \
    --snap=10 \
    --metrics=none

print("Training complete. Check results in:", RESULTS_DIR)

## Part 11.5 — Load Fine-Tuned Checkpoint

After training, we load the latest checkpoint for generation.
If training was interrupted, we load the most recent `.pkl` file saved.

In [ ]:
import glob, pickle

# Find the latest checkpoint in results directory
checkpoints = sorted(glob.glob(os.path.join(RESULTS_DIR, "**", "*.pkl"), recursive=True))

if not checkpoints:
    print("No fine-tuned checkpoint found. Using pre-trained FFHQ checkpoint for demo.")
    FINETUNED_PATH = PRETRAINED_PATH
else:
    FINETUNED_PATH = checkpoints[-1]
    print(f"Found {len(checkpoints)} checkpoint(s).")
    print(f"Using latest: {FINETUNED_PATH}")

# Load the generator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(FINETUNED_PATH, "rb") as f:
    data = pickle.load(f)

G = data["G_ema"].to(device).eval()   # G_ema = exponential moving average — smoother outputs

print("Generator loaded successfully.")
print("Latent dimension (Z space):", G.z_dim)
print("Mapping network output (W space):", G.w_dim)

## Part 11.6 — Emotion Conditioning via W Latent Space

**بالعربي البسيط:** الـ Generator مش بياخد الصورة مباشرة —
بياخد vector عشوائي (Z space) وبيحوله لـ style vector (W space)
اللي بيتحكم في كل تفاصيل الصورة.

**Emotion Conditioning** — عندنا 3 طرق:

| الطريقة | المعنى |
|---------|--------|
| **Seed selection** | لكل emotion بنختار seeds معينة أعطت نتائج مناسبة |
| **W vector shifting** | بنزود أو بننقص dimensions معينة في الـ W vector حسب الـ emotion |
| **Truncation trick** | بنتحكم في مدى قرب الصورة من المتوسط (أعلى = أوضح، أقل = أكثر تنوعاً) |

هنستخدم الـ **W vector shifting** — هو الأبسط والأكثر تحكماً.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Emotion direction vectors in W space
# These shift the latent vector toward visual properties of each emotion
# Positive = brighter/warmer/more energetic, Negative = darker/cooler/calmer
EMOTION_DIRECTIONS = {
    "Joy"    : {"brightness_shift": +0.8,  "truncation": 0.6},
    "Calm"   : {"brightness_shift": +0.1,  "truncation": 0.5},
    "Sadness": {"brightness_shift": -0.8,  "truncation": 0.7},
}

# Artist-specific seed ranges — seeds that produced artwork-like outputs
# These are found by sampling and selecting the best seeds per artist
ARTIST_SEEDS = {
    "Van Gogh"         : list(range(100, 150)),
    "Monet"            : list(range(200, 250)),
    "Leonardo da Vinci": list(range(300, 350)),
}

def generate_from_dataset_model(
    artist         : str,
    emotion        : str,
    seed           : int  = None,
    truncation_psi : float = None,
) -> Image.Image:
    """
    Generate an artwork image using the fine-tuned StyleGAN2-ADA generator.

    Parameters
    ----------
    artist         : One of 'Van Gogh', 'Monet', 'Leonardo da Vinci'
    emotion        : One of 'Joy', 'Calm', 'Sadness'
    seed           : Random seed — if None, picks randomly from artist seed range
    truncation_psi : Controls diversity vs quality (0.5 = high quality, 1.0 = max diversity)

    Returns
    -------
    PIL Image
    """
    if seed is None:
        seed = np.random.choice(ARTIST_SEEDS[artist])

    emotion_cfg    = EMOTION_DIRECTIONS[emotion]
    truncation_psi = truncation_psi or emotion_cfg["truncation"]

    # Sample Z vector
    z = torch.from_numpy(
        np.random.RandomState(seed).randn(1, G.z_dim)
    ).to(device).float()

    # Map Z → W
    with torch.no_grad():
        w = G.mapping(z, None, truncation_psi=truncation_psi)

        # Apply emotion shift to W space
        # We shift the first 4 dimensions which influence overall brightness/mood
        shift_amount  = emotion_cfg["brightness_shift"]
        w[:, :, :4]  += shift_amount

        # Generate image from W
        img_tensor = G.synthesis(w, noise_mode="const")

    # Convert tensor to PIL Image
    # Tensor is in [-1, 1], convert to [0, 255]
    img_tensor = (img_tensor.clamp(-1, 1) + 1) / 2 * 255
    img_np     = img_tensor[0].permute(1, 2, 0).cpu().numpy().astype(np.uint8)
    return Image.fromarray(img_np)


print("Generation function defined.")
print("Ready to generate artworks from our fine-tuned model.")

## Part 11.7 — Emotion Detection from User Text

Same emotion detection pipeline from before:
User text (Arabic or English) → Translation if needed → Emotion classifier → Joy / Calm / Sadness

In [ ]:
from transformers import pipeline as hf_pipeline

print("Loading translation model (Arabic → English)...")
translator = hf_pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-ar-en",
    device=0 if torch.cuda.is_available() else -1
)

print("Loading emotion classifier...")
emotion_classifier = hf_pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device=0 if torch.cuda.is_available() else -1
)

EMOTION_MAP = {
    "joy"    : "Joy",     "surprise": "Joy",
    "neutral": "Calm",    "disgust"  : "Calm",
    "fear"   : "Sadness", "sadness"  : "Sadness", "anger": "Sadness",
}

AUTO_ARTIST_MAP = {
    "Joy"    : "Van Gogh",
    "Calm"   : "Monet",
    "Sadness": "Leonardo da Vinci",
}

def detect_emotion(user_text: str) -> dict:
    is_arabic = any("\u0600" <= c <= "\u06FF" for c in user_text)
    if is_arabic:
        translated      = translator(user_text, max_length=128)[0]["translation_text"]
        text_for_model  = translated
        print(f"Translated: '{user_text}' → '{translated}'")
    else:
        text_for_model = user_text

    results   = emotion_classifier(text_for_model)[0]
    top       = max(results, key=lambda x: x["score"])
    raw_label = top["label"].lower()
    mapped    = EMOTION_MAP.get(raw_label, "Calm")

    return {
        "input_text"   : user_text,
        "emotion_label": mapped,
        "confidence"   : round(top["score"], 4),
        "raw_emotion"  : raw_label,
    }

print("Emotion detection ready.")

## Part 11.8 — Claude AI Artwork Description

After generating the image, we send it to Claude API.
Claude analyzes the visual elements and writes a detailed interpretation
connecting artistic choices to the user's emotion.

Get your API key at: https://console.anthropic.com

In [ ]:
import anthropic, base64, io

ANTHROPIC_API_KEY = "your-api-key-here"
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def image_to_base64(image: Image.Image) -> str:
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return base64.standard_b64encode(buffer.getvalue()).decode("utf-8")

def describe_artwork(image: Image.Image, artist: str, emotion: str, user_text: str) -> str:
    img_b64 = image_to_base64(image)

    system_prompt = (
        "You are an expert art critic and emotion analyst. "
        "You analyze AI-generated artworks inspired by famous painters "
        "and explain how their visual elements reflect specific human emotions. "
        "Your descriptions are vivid, specific, and emotionally resonant. "
        "Always connect visual details (colors, brushstrokes, composition, light) "
        "directly to the emotional experience."
    )

    user_prompt = (
        f"The user expressed: '{user_text}'\n"
        f"Detected emotion: {emotion}\n"
        f"Generated in the style of: {artist}\n"
        f"This image was generated by a StyleGAN2 model fine-tuned on real artworks "
        f"by {artist} from the Best Artworks of All Time dataset.\n\n"
        f"Please analyze this generated artwork in detail. Explain:\n"
        f"1. The dominant colors and what emotional role they play\n"
        f"2. The brushwork, texture, or visual patterns and how they reflect {emotion}\n"
        f"3. The overall composition and mood\n"
        f"4. How this image connects to what the user expressed: '{user_text}'\n\n"
        f"Write 3-4 paragraphs. Be specific about what you see in the image."
    )

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=600,
        system=system_prompt,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {"type": "base64", "media_type": "image/png", "data": img_b64},
                },
                {"type": "text", "text": user_prompt},
            ],
        }],
    )
    return response.content[0].text

print("Claude description function ready.")

## Part 11.9 — Full Pipeline: Text → Emotion → Generate → Describe

In [ ]:
def run_full_pipeline(
    user_text : str,
    artist    : str  = None,
    seed      : int  = None,
) -> dict:
    """
    Full end-to-end pipeline.

    Parameters
    ----------
    user_text : What the user typed — Arabic or English
    artist    : 'Van Gogh', 'Monet', 'Leonardo da Vinci', or None for auto-select
    seed      : Fixed seed for reproducibility, or None for random
    """
    print("=" * 60)
    print("STEP 1 — Detecting emotion from user input")
    print("=" * 60)
    emotion_result = detect_emotion(user_text)
    emotion_label  = emotion_result["emotion_label"]
    print(f"Emotion  : {emotion_label}  (raw: {emotion_result['raw_emotion']}, confidence: {emotion_result['confidence']})")

    if artist is None:
        artist = AUTO_ARTIST_MAP[emotion_label]
        print(f"Auto-selected artist: {artist}")
    else:
        print(f"Artist (user selected): {artist}")

    print()
    print("=" * 60)
    print("STEP 2 — Generating artwork from fine-tuned StyleGAN2-ADA")
    print("=" * 60)
    image = generate_from_dataset_model(artist, emotion_label, seed=seed)

    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.title(f"{artist}  |  {emotion_label}", fontsize=13, fontweight="bold", pad=10)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    print()
    print("=" * 60)
    print("STEP 3 — Claude AI artwork interpretation")
    print("=" * 60)
    description = describe_artwork(image, artist, emotion_label, user_text)
    print()
    print(description)
    print()

    return {"emotion": emotion_result, "artist": artist, "image": image, "description": description}


# ── Run examples ──
r1 = run_full_pipeline("أنا حاسس بحزن شديد وكل حاجة تقيلة", artist=None, seed=42)

In [ ]:
r2 = run_full_pipeline("I feel so peaceful and content today", artist="Monet", seed=200)

In [ ]:
r3 = run_full_pipeline("أنا سعيد جداً النهارده", artist="Van Gogh", seed=110)

## Part 11.10 — Generate All 9 Combinations Grid

Generate one image per artist × emotion combination (9 total).
This is your showcase grid for LinkedIn and GitHub.

In [ ]:
os.makedirs("/content/final_artworks", exist_ok=True)

ARTISTS  = ["Van Gogh", "Monet", "Leonardo da Vinci"]
EMOTIONS = ["Joy", "Calm", "Sadness"]
EMOTION_COLORS = {"Joy": "#DAA520", "Calm": "#4682B4", "Sadness": "#6A5ACD"}

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
fig.suptitle(
    "Emotion-Aware Art Generation\nStyleGAN2-ADA Fine-Tuned on Best Artworks Dataset",
    fontsize=15, fontweight="bold", y=1.01
)

for row_idx, emotion in enumerate(EMOTIONS):
    for col_idx, artist in enumerate(ARTISTS):
        seed = row_idx * 50 + col_idx * 10 + 100
        img  = generate_from_dataset_model(artist, emotion, seed=seed)

        fname = f"/content/final_artworks/{artist.replace(' ', '_')}_{emotion}.png"
        img.save(fname)

        axes[row_idx][col_idx].imshow(img)
        axes[row_idx][col_idx].set_title(
            f"{artist}\n{emotion}",
            fontsize=9, fontweight="bold",
            color=EMOTION_COLORS[emotion]
        )
        axes[row_idx][col_idx].axis("off")

plt.tight_layout()
grid_path = "/content/final_artworks/showcase_grid.png"
plt.savefig(grid_path, dpi=150, bbox_inches="tight")
plt.show()
print("Showcase grid saved to:", grid_path)

## Part 11.11 — Interactive Demo Widget

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

text_input = widgets.Textarea(
    placeholder="Type how you feel in Arabic or English...",
    layout=widgets.Layout(width="500px", height="80px")
)

artist_dropdown = widgets.Dropdown(
    options=["Auto (AI decides)", "Van Gogh", "Monet", "Leonardo da Vinci"],
    description="Artist:",
    style={"description_width": "initial"},
)

seed_slider = widgets.IntSlider(
    value=42, min=0, max=500, step=1,
    description="Seed:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px")
)

generate_btn = widgets.Button(
    description="Generate Artwork",
    button_style="primary",
    layout=widgets.Layout(width="200px", height="40px")
)

output_area = widgets.Output()

def on_generate(btn):
    with output_area:
        clear_output(wait=True)
        user_text = text_input.value.strip()
        if not user_text:
            print("Please type something first.")
            return
        selected_artist = None if artist_dropdown.value == "Auto (AI decides)" \
                          else artist_dropdown.value
        run_full_pipeline(
            user_text=user_text,
            artist=selected_artist,
            seed=seed_slider.value
        )

generate_btn.on_click(on_generate)

display(widgets.VBox([
    widgets.HTML("<h3>Emotion-Aware Art Generator — StyleGAN2-ADA Fine-Tuned on Artworks Dataset</h3>"),
    widgets.HTML("<b>How are you feeling?</b>"),
    text_input,
    artist_dropdown,
    seed_slider,
    generate_btn,
    output_area,
]))

---
## Complete Pipeline Summary

| Stage | Method | Output |
|-------|--------|--------|
| EDA | Full dataset exploration — 50 artists, 8683 images | Charts, insights |
| Filtering | Keep Van Gogh, Monet, Leonardo da Vinci | Filtered DataFrame |
| Feature Engineering | 14 pixel stats + metadata + derived features | Feature DataFrame |
| Emotion Labeling | Color-based heuristics + optional CLIP | emotion_final column |
| GAN Preprocessing | Resize 256x256, normalize [-1,1], augmentation | gan_dataset/ folder |
| Transfer Learning | StyleGAN2-ADA fine-tuned from FFHQ checkpoint | Fine-tuned generator |
| Emotion Conditioning | W latent space shifting per emotion | Emotion-aware generation |
| User Input | Arabic/English text → emotion detection | Joy / Calm / Sadness |
| Generation | Fine-tuned GAN generates new artwork | PIL Image |
| AI Description | Claude API analyzes and interprets image | Detailed text description |
| Demo | Interactive ipywidgets interface | Live demo |

**Next steps after this notebook:**
- Evaluate generation quality using FID score
- Build a Streamlit web app for public deployment
- Add more artists and emotion categories